# embodied-efficiency — VLA flow-matching kernel on a free T4

**Runtime → Change runtime type → T4 GPU** (free), then *Run all*.

Produces: the fp16 baseline (eager vs CUDA-graphs), the INT8/INT4 fused-kernel latency, and a kernel-correctness check. Free/unbilled — use the standard **T4** only, never a premium accelerator.

In [ ]:
import torch
print('torch', torch.__version__)
print('gpu  ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set runtime to T4')
try:
    import triton; print('triton', triton.__version__)
except ImportError:
    !pip -q install triton

In [ ]:
# Edit REPO if your GitHub user/name differs. For a private repo, clone with a token.
REPO = 'https://github.com/LaelaZorana/embodied-efficiency.git'
!git clone -q $REPO
%cd embodied-efficiency

## 1. fp16 baseline — eager vs CUDA-graphs (the baseline that matters)

In [ ]:
!python kernel/bench.py --steps 10 4 8 --batch 1 --dtype fp16 --compile --peak T4

## 2. INT8 and INT4 fused kernels (the win)

In [ ]:
!python kernel/bench.py --steps 10 4 8 --batch 1 --dtype fp16 --quant int8 --peak T4
!python kernel/bench.py --steps 10 4 8 --batch 1 --dtype fp16 --quant int4 --peak T4

## 3. Kernel correctness (triton vs torch reference) + fidelity

In [ ]:
!python kernel/triton_gemm.py
!python kernel/quant.py

## What to read off
- `compile_reduce_overhead` ms/step = strong baseline (CUDA graphs already kill launch overhead).
- `int8/eager` and `int4/eager` ms/step → realized speedup; compare to the **1.97× / 3.88×** weight-traffic ceilings.
- correctness: small max-abs-err + `Triton kernel correctness ✓`, and action rMSE 0.0025 (int8) / 0.0423 (int4).
- combining the kernels with CUDA graphs is the next optimization (orthogonal wins).